In [1]:
import os
import sys
os.chdir("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [2]:
df_listings = pd.read_csv('data/combined_csvs/listings_property_vals.csv')


In [3]:
# df_listings["zip3"] = df_listings["zipcode"].apply(lambda x: str(x)[0:3])

city_centers = {
    'raleigh': (35.7796, -78.6382),
    'charlotte': (35.2271, -80.8431),
    'durham': (35.9940, -78.8986),
    'asheville': (35.5951, -82.5515),
    'wilmington': (34.2104, -77.9447),
    'carolina beach': (34.0352, -77.8936),
    'myrtle beach': (33.6891, -78.8867),
    'pigeon forge': (35.7884, -83.5543),
    'gatlinburg': (35.7143, -83.5102),
    'williamsburg': (37.2695, -76.7084)
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates the distance in miles between two lat/lon points."""
    # Convert decimal degrees to radians 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    
    # 3956 is the radius of the Earth in miles. (Use 6371 for kilometers)
    miles = 3956 * c
    return miles

In [4]:
# Feature engineer distance to city center

# 1. Standardize the city column to match the dictionary keys
df_listings['city_lower'] = df_listings['city'].str.lower().str.strip()

# 2. Map the center coordinates to each row based on its city
df_listings['center_lat'] = df_listings['city_lower'].map(lambda x: city_centers.get(x, (np.nan, np.nan))[0])
df_listings['center_lon'] = df_listings['city_lower'].map(lambda x: city_centers.get(x, (np.nan, np.nan))[1])

# 3. Calculate the distance!
df_listings['dist_to_center_miles'] = haversine_distance(
    df_listings['latitude'], 
    df_listings['longitude'], 
    df_listings['center_lat'], 
    df_listings['center_lon']
)

# 4. Clean up the temporary columns
df_listings = df_listings.drop(columns=['city_lower', 'center_lat', 'center_lon'])

In [ ]:
X_col = [
    "property_val",
    "room_type",
    "bedrooms",
    "baths",
    "ttm_available_days",
    "ttm_blocked_days",
    "SizeRank",
    "city",
    "dist_to_center_miles"
]
Y_col = ["ttm_revenue"]

In [6]:
def calculate_gvif(df, features):
    """
    Automatically preprocesses categorical data and calculates GVIF.
    
    Parameters:
    df: Raw pandas DataFrame (can contain text/objects).
    features: List of column names to include in the model.
              e.g., ['property_val', 'beds', 'room_type', 'zipcode']
    """
    # 1. Isolate only the requested features and drop missing values
    # (GVIF requires a complete matrix without NaNs)
    df_subset = df[features].dropna()
    
    # 2. Identify numeric vs categorical columns
    numeric_cols = df_subset.select_dtypes(include=['number', 'bool']).columns.tolist()
    cat_cols = df_subset.select_dtypes(exclude=['number', 'bool']).columns.tolist()
    
    # 3. One-hot encode categorical variables automatically
    # dtype=float prevents pandas boolean issues in numpy matrices
    df_encoded = pd.get_dummies(df_subset, columns=cat_cols, drop_first=True, dtype=float)
    
    # 4. Build the column mapping internally
    feature_groups = {}
    for col in numeric_cols:
        feature_groups[col] = [col]
        
    for col in cat_cols:
        # Find all dummy columns generated for this original categorical column
        dummy_cols = [c for c in df_encoded.columns if c.startswith(f"{col}_")]
        if dummy_cols: 
            feature_groups[col] = dummy_cols

    # 5. Calculate GVIF
    corr_matrix = df_encoded.corr().values
    det_all = np.linalg.det(corr_matrix)
    
    # Safety check for perfect multicollinearity
    if np.isclose(det_all, 0):
        raise ValueError("Perfect multicollinearity detected. Determinant is 0. Check your features for duplicates or exact linear combinations.")
    
    results = []
    
    for feature_name, columns in feature_groups.items():
        # Get integer indices of the columns for this feature block
        col_indices = [df_encoded.columns.get_loc(col) for col in columns]
        
        # Get integer indices of all OTHER columns
        other_indices = [i for i in range(df_encoded.shape[1]) if i not in col_indices]
        
        # Determinants
        if len(col_indices) > 1:
            det_feature = np.linalg.det(df_encoded.iloc[:, col_indices].corr().values)
        else:
            det_feature = 1.0 
            
        det_other = np.linalg.det(df_encoded.iloc[:, other_indices].corr().values)
        
        # Calculate GVIF
        gvif = (det_feature * det_other) / det_all
        
        # Calculate Degrees of Freedom
        df_val = len(col_indices)
        
        # Calculate Adjusted GVIF
        gvif_adjusted = gvif ** (1 / (2 * df_val))
        
        results.append({
            'Feature': feature_name,
            'GVIF': gvif,
            'df': df_val,
            'GVIF^(1/2df)': gvif_adjusted
        })
        
    # Return results sorted by the adjusted GVIF
    return pd.DataFrame(results).sort_values(by='GVIF^(1/2df)', ascending=False)

In [7]:
calculate_gvif(df_listings, X_col)

,Feature,GVIF,df,GVIF^(1/2df)
1,bedrooms,5.661974,1,2.379490
0,property_val,4.653543,1,2.157207
5,SizeRank,3.794213,1,1.947874
2,baths,3.468145,1,1.862296
3,ttm_available_days,1.295212,1,1.138074
6,dist_to_center_miles,1.274188,1,1.128799
8,city,6.196091,9,1.106640
4,ttm_blocked_days,1.188587,1,1.090223
7,room_type,1.423197,3,1.060582


In [8]:
# filter out rows with missing target value
df_listings = df_listings[~pd.isna(df_listings["gross_yield"])]

y = df_listings[Y_col]
X = df_listings[X_col]

# set str columns as categories
categorical_cols = ["room_type", "city"]
X[categorical_cols] = X[categorical_cols].astype("category")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist",  # Fast histogram optimized training
    enable_categorical=True,
    early_stopping_rounds=50,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,  # Prints log every 50 trees
)

preds = model.predict(X_test)
rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"\nModel Performance:")
print(f"Best Iteration (Tree Count): {model.best_iteration}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² Score: {r2:.4f}")

importance = pd.DataFrame(
    {"Feature": X.columns, "Importance": model.feature_importances_}
).sort_values(by="Importance", ascending=False)
print("\nFeature Importance:")
print(importance.to_string(index=False))

[0]	validation_0-rmse:0.06810
[50]	validation_0-rmse:0.03574
[100]	validation_0-rmse:0.03396
[150]	validation_0-rmse:0.03350
[200]	validation_0-rmse:0.03345
[232]	validation_0-rmse:0.03355

Model Performance:
Best Iteration (Tree Count): 182
RMSE: 0.0334
MAE: 0.0185
R² Score: 0.7740

Feature Importance:
             Feature  Importance
  ttm_available_days    0.367651
                city    0.185722
           room_type    0.155071
               baths    0.059854
        property_val    0.058471
            SizeRank    0.054326
    ttm_blocked_days    0.041380
            bedrooms    0.039143
dist_to_center_miles    0.038381
